# WBSNet — Option-A Paper Run on Colab

End-to-end runner for the **Option-A paper scope**:

- 7 ablation variants (A1-A7) × seeds on **Kvasir-SEG**
- Full WBSNet on **CVC-ClinicDB** and **ISIC2018**
- U-Net baselines on Kvasir, ClinicDB, ISIC2018
- Generalization eval: Kvasir checkpoint → CVC-ColonDB
- Aggregation, significance tests, complexity, qualitative figures

**Recommended runtime:** Colab Pro+ A100 with background execution.
**Backup:** Colab Pro L4 for smoke tests or smaller runs.

Before running:
1. `Runtime → Change runtime type → A100 GPU` (or L4 if on Pro).
2. `Runtime → Background execution` (Pro+ only) — lets you close the laptop.
3. Have the processed dataset on Google Drive at `MyDrive/WBSNet_Dataset/` with subdirs `kvasir/`, `cvc_clinicdb/`, `cvc_colondb/`, `isic2018/`.
4. If continuing from the Kaggle seed-3407 run, keep the exported `wbsnet_paper_runs/` folder at `MyDrive/wbsnet_paper_runs/`.
5. The expected processed layout is split-based: `kvasir/train/images`, `kvasir/val/masks`, `isic2018/test/images`, `cvc_colondb/test/masks`, etc.

Each section is independent — re-run only what you need.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Environment check

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Switch runtime to A100 (Pro+) or L4 (Pro).")

gpu_name = torch.cuda.get_device_name(0)
print(f"\nDetected GPU: {gpu_name}")
SUPPORTED = ("A100", "L4", "H100")
if not any(tag in gpu_name for tag in SUPPORTED):
    raise RuntimeError(
        f"Unsupported GPU '{gpu_name}'. The paper pipeline targets A100/L4/H100. "
        "Change via Runtime -> Change runtime type before running the paper pipeline."
    )


## 2. Clone repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MrArrogant2002/WBSNet-research-paper.git"
REPO_DIR = Path("/content/WBSNet-research-paper")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin", "main"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)

print("CWD:", os.getcwd())
print("Runtime repo commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)


## 3. Install dependencies

Colab already ships with CUDA-enabled PyTorch. Install the Colab-compatible dependency set, then install this repo with `--no-deps` so editable install does not downgrade the runtime image.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# This cell is self-checking because Colab can lose Python state after reconnects.
REPO_DIR = Path(globals().get("REPO_DIR", "/content/WBSNet-research-paper"))
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository not found at {REPO_DIR}. Run Section 2 (Clone repo) first, "
        "then rerun this install cell."
    )

os.chdir(REPO_DIR)
requirements = REPO_DIR / "requirements-colab.txt"
if not requirements.exists():
    raise FileNotFoundError(f"Missing {requirements}; current directory is {Path.cwd()}")

print("CWD:", Path.cwd())
print("Installing Colab runtime dependencies from", requirements)
# Keep Colab's CUDA-enabled PyTorch. Do not install torch/torchvision here.
# Do not run quiet: if pip fails, we need the exact resolver/build error.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)], check=True)

import torch
print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 4. Verify repo structure

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/verify_repo.py"], check=True)


## 5. Mount Google Drive and link dataset

If your dataset is laid out differently, edit `DRIVE_DATASET` and `DATASET_MAP` below.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_DATASET = Path("/content/drive/MyDrive/WBSNet_Dataset")
assert DRIVE_DATASET.exists(), f"Dataset not found at {DRIVE_DATASET}. Edit DRIVE_DATASET in this cell."

DATA_ROOT = REPO_DIR / "data"
DATA_ROOT.mkdir(exist_ok=True)

DATASET_MAP = {
    "kvasir": "Kvasir-SEG",
    "cvc_clinicdb": "CVC-ClinicDB",
    "cvc_colondb": "CVC-ColonDB",
    "isic2018": "ISIC2018",
}

for src_name, dst_name in DATASET_MAP.items():
    src = DRIVE_DATASET / src_name
    dst = DATA_ROOT / dst_name
    if not src.exists():
        print(f"MISSING: {src}")
        continue
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        print(f"EXISTS: {dst} is not a symlink; leaving it unchanged")
        continue
    os.symlink(src, dst)
    print(f"LINKED: {src} -> {dst}")

print("\nDataset directory links:")
for item in sorted(DATA_ROOT.iterdir()):
    if item.is_symlink():
        print(f"{item} -> {item.resolve()}")


## 6. Restore previous outputs and import legacy / Kaggle-session artifacts

Run this **before** the paper pipeline. It pulls every prior result that lives on Drive into `outputs/` so already-completed runs are skipped instead of retrained.

This cell supports both Kaggle layouts:

| Drive layout | What it holds |
|---|---|
| `paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>/` | Legacy expanded run folders |
| `archives/paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>.tar.gz` | New archive-first Kaggle output, used to avoid Drive API quota stalls |

| Drive root | What it holds |
|---|---|
| `wbsnet_paper_runs/` | Original Kaggle seed-3407 - Kvasir A1-A7, CVC-ClinicDB A1-A7, ISIC2018 A1 |
| `wbsnet_kaggle_session1/` | Kaggle Session 1 - ISIC2018 A2 seed 3407 |
| `wbsnet_kaggle_session2/` | Kaggle Session 2 - Kvasir A2/A5/A6/A7 seed 3408 |
| `wbsnet_kaggle_session3/` | Kaggle Session 3 - Kvasir A1/A3/A4 + ClinicDB A1/A2 seed 3408 |
| `WBSNet_outputs/` | Any partial Colab run from a previous session |

See `kaggle-session-plan.md` in the repo root for the strategy behind the three Kaggle sessions and what work remains for this Colab notebook to handle.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tarfile

DRIVE_OUTPUTS = Path("/content/drive/MyDrive/WBSNet_outputs")
EXTRACTED_KAGGLE_ARCHIVES = Path("/content/wbsnet_kaggle_archives")

# All Drive roots that may hold importable runs in either the legacy
# `paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>/` layout or the new
# `archives/paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>.tar.gz` layout.
# Order matters: later entries can overwrite earlier ones if they contain
# the same run, so list newer / higher-quality sources later.
LEGACY_ROOTS = [
    Path("/content/drive/MyDrive/wbsnet_paper_runs"),       # original Kaggle seed-3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session1"),  # ISIC2018 A2 seed 3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session2"),  # Kvasir seed 3408 subset
    Path("/content/drive/MyDrive/wbsnet_kaggle_session3"),  # Kvasir/ClinicDB seed 3408 subset
]


def run_checked(cmd):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=True)


def safe_extract_tar(archive_path: Path, target_dir: Path):
    target_root = target_dir.resolve()
    with tarfile.open(archive_path, "r:gz") as tar:
        members = tar.getmembers()
        for member in members:
            member_target = (target_dir / member.name).resolve()
            if target_root not in [member_target, *member_target.parents]:
                raise RuntimeError(f"Unsafe archive path in {archive_path}: {member.name}")
        tar.extractall(target_dir, members=members)


def extract_kaggle_archives(legacy_root: Path):
    archives_root = legacy_root / "archives" / "paper_suite"
    if not archives_root.exists():
        return None
    archives = sorted(archives_root.rglob("*.tar.gz"))
    if not archives:
        return None

    extracted_root = EXTRACTED_KAGGLE_ARCHIVES / legacy_root.name
    extracted_paper_suite = extracted_root / "paper_suite"
    extracted_paper_suite.mkdir(parents=True, exist_ok=True)

    print(f"  extracting {len(archives)} archive(s) from {archives_root}")
    for archive_path in archives:
        rel_parent = archive_path.parent.relative_to(archives_root)
        target_parent = extracted_paper_suite / rel_parent
        target_parent.mkdir(parents=True, exist_ok=True)
        marker = target_parent / f".{archive_path.name}.extracted"
        marker_value = f"{archive_path.stat().st_size}:{archive_path.stat().st_mtime_ns}"
        if marker.exists() and marker.read_text(encoding="utf-8") == marker_value:
            continue
        print(f"    extract: {archive_path.name}")
        safe_extract_tar(archive_path, target_parent)
        marker.write_text(marker_value, encoding="utf-8")
    return extracted_root


def importable_roots(legacy_root: Path):
    roots = []
    if (legacy_root / "paper_suite").exists():
        roots.append(legacy_root)
    extracted_root = extract_kaggle_archives(legacy_root)
    if extracted_root is not None and (extracted_root / "paper_suite").exists():
        roots.append(extracted_root)
    return roots


# 1. Restore any previous Colab output backup (in-progress checkpoints, etc.).
if DRIVE_OUTPUTS.exists():
    print(f"Restoring previous Colab outputs from {DRIVE_OUTPUTS}")
    Path("outputs").mkdir(exist_ok=True)
    run_checked(["rsync", "-a", "--info=progress2", f"{DRIVE_OUTPUTS}/", "outputs/"])
else:
    print(f"No previous Colab output backup found at {DRIVE_OUTPUTS}")

# 2. Import every legacy / Kaggle-session root that exists. The importer
#    is idempotent - re-running just refreshes metadata for runs that are
#    already present.
imported_any = False
for legacy_root in LEGACY_ROOTS:
    if not legacy_root.exists():
        print(f"  skip: {legacy_root} not present on Drive")
        continue

    roots = importable_roots(legacy_root)
    if not roots:
        print(f"  skip: {legacy_root} has no paper_suite folder or archives")
        continue

    for import_root in roots:
        for seed in (3407, 3408):
            print(f"\nImporting seed {seed} from {import_root}")
            run_checked([
                sys.executable,
                "scripts/import_legacy_paper_runs.py",
                "--legacy-root", str(import_root),
                "--output-root", "outputs",
                "--seed", str(seed),
                "--allow-missing",
                "--verify-forward",
            ])
            imported_any = True

if not imported_any:
    print("\nNo legacy / Kaggle-session folders or archives found on Drive; starting from scratch.")

# 3. List every imported best.pt so the user can confirm what is now skippable.
print("\nExisting best checkpoints that will be skipped by the paper runner:")
checkpoints = sorted(Path("outputs").glob("**/checkpoints/best.pt")) if Path("outputs").exists() else []
for ckpt in checkpoints[:120]:
    print(ckpt)
print(f"Total best checkpoints found: {len(checkpoints)}")


## 7. (Optional) W&B login

Skip this cell to run with W&B offline. Set `WANDB_API_KEY` to log online.

In [ ]:
import os

# Preferred: store the key once via the key icon in the left sidebar (Colab Secrets)
# as 'WANDB_API_KEY', then load it here. Falls back gracefully if the secret is absent.
try:
    from google.colab import userdata  # type: ignore[import-not-found]

    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        print("WANDB_API_KEY loaded from Colab Secrets.")
    else:
        print("Colab Secret 'WANDB_API_KEY' not set; W&B will run offline unless you set it below.")
except Exception as exc:  # noqa: BLE001
    print(f"Colab userdata unavailable ({exc}); W&B will run offline unless you set it below.")

# Alternative paths (uncomment one if you do not want to use Colab Secrets):
#  - Hardcode for a single session (NOT recommended; never commit this):
#       os.environ["WANDB_API_KEY"] = "YOUR_WANDB_KEY_HERE"
#  - Place a `.env` file at the repo root with WANDB_API_KEY=...; ExperimentLogger auto-loads it.
#  - Interactive login that does not echo the key:
#       from getpass import getpass; os.environ["WANDB_API_KEY"] = getpass("WANDB key: ")
# Then, to actually log in to the W&B CLI for this session:
# !wandb login --relogin


## 8. Smoke test (1 epoch, batch 2)

Confirms the pipeline runs end-to-end. ~2 min on A100, ~3 min on L4.

In [ ]:
import subprocess
import sys

# Smoke test: 1 epoch, tiny batch. Run this before the full paper run.
cmd = [
    sys.executable, "train.py",
    "--config", "configs/kvasir_wbsnet.yaml",
    "--override",
    "train.epochs=1",
    "train.batch_size=2",
    "train.grad_accum_steps=1",
    "dataset.split_strategy=pre_split_dirs",
    "dataset.num_workers=2",
    "dataset.prefetch_factor=2",
    "evaluation.compute_hd95=false",
    "evaluation.max_visualizations=0",
    "runtime.device=cuda",
    "runtime.wandb.mode=offline",
    "experiment.name=smoke_test",
    "experiment.run_name=smoke_test",
]
subprocess.run(cmd, check=True)


## 9. Run the Option-A controller for seeds 3407 and 3408

This is the default paper-run cell for the current budget plan. It assumes the Kaggle/legacy outputs from Section 6 have already been imported.

The cell runs `scripts/run_paper_optionA.py --seeds 3407 3408`. Imported `checkpoints/best.pt` files are skipped, so the only training that should actually run after all three Kaggle sessions complete is:

| Seed | Dataset | Run | Config |
|---|---|---|---|
| 3408 | ISIC2018 | A1 baseline / U-Net | `configs/isic2018_unet_baseline.yaml` |
| 3408 | ISIC2018 | A2 full WBSNet | `configs/isic2018_wbsnet.yaml` |

Before launching the controller, the cell checks that the expected Kaggle/legacy checkpoints are present. Leave `ALLOW_OPTIONA_WITH_MISSING_IMPORTS = False` unless you intentionally want Colab to train missing Kaggle work too.

Do **not** include seed 3409 here unless you intentionally choose the expensive n=3 plan later.

In [ ]:
# Colab continuation run. Imported checkpoints are skipped by run_paper_optionA;
# after all Kaggle sessions complete, only ISIC2018 A1/A2 seed 3408 may train.

import os
import re
import signal
import subprocess
import sys
from pathlib import Path

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

ALLOW_OPTIONA_WITH_UNEXPECTED_TRAINING = False
AUDIT_ONLY = False  # Set True to print the cost audit and stop before A100 work.
ALLOWED_COLAB_TRAIN_CHECKPOINTS = {
    Path("outputs/isic2018_unet_baseline/isic2018_unet_baseline_seed3408/checkpoints/best.pt"),
    Path("outputs/isic2018_wbsnet/isic2018_wbsnet_seed3408/checkpoints/best.pt"),
}

COMMON_OVERRIDES = [
    "train.epochs=150",
    "train.batch_size=8",
    "train.grad_accum_steps=2",
    "train.encoder_lr=0.00005",
    "train.decoder_lr=0.0005",
    "train.nonfinite_grad_action=skip",
    "train.max_nonfinite_grad_steps=10",
    "dataset.split_strategy=pre_split_dirs",
    "dataset.num_workers=2",
    "dataset.prefetch_factor=2",
    "train.save_every=5",
    "runtime.device=cuda",
    "runtime.wandb.mode=offline",
    "evaluation.max_visualizations=8",
]


def run_checked(cmd):
    print("\n>>> " + " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(x) for x in cmd], check=True)

def patch_significance_tests():
    script = Path("scripts/significance_tests.py")
    text = script.read_text(encoding="utf-8")
    if "def _metric_by_seed" in text:
        print("significance_tests.py already has duplicate-seed handling.")
        return

    replacement = '''def _metric_by_seed(frame: pd.DataFrame, metric: str) -> pd.Series:
    data = frame[["seed", metric]].dropna(subset=["seed", metric]).copy()
    if data.empty:
        return pd.Series(dtype="float64")
    data[metric] = pd.to_numeric(data[metric], errors="coerce")
    data = data.dropna(subset=[metric])
    if data.empty:
        return pd.Series(dtype="float64")
    # Some runs can produce multiple evaluation records for the same seed.
    # Collapse those records before paired tests so each seed contributes once.
    return data.groupby("seed", sort=True)[metric].mean()


def _paired_or_independent_test(frame_a: pd.DataFrame, frame_b: pd.DataFrame, metric: str) -> dict[str, Any]:
    data_a = _metric_by_seed(frame_a, metric)
    data_b = _metric_by_seed(frame_b, metric)
    if not data_a.empty and not data_b.empty:
        shared = sorted(set(data_a.index) & set(data_b.index))
        if len(shared) >= 2:
            paired_a = data_a.loc[shared].astype(float)
            paired_b = data_b.loc[shared].astype(float)
            statistic, p_value = stats.ttest_rel(paired_a.to_numpy(), paired_b.to_numpy())
            return {
                "test_type": "paired_t_test",
                "statistic": float(statistic),
                "p_value": float(p_value),
                "n_compared": int(len(shared)),
            }

    values_a = data_a.dropna().astype(float)
    values_b = data_b.dropna().astype(float)
    if len(values_a) >= 2 and len(values_b) >= 2:
        statistic, p_value = stats.ttest_ind(values_a.to_numpy(), values_b.to_numpy(), equal_var=False)
        return {
            "test_type": "welch_t_test",
            "statistic": float(statistic),
            "p_value": float(p_value),
            "n_compared": int(min(len(values_a), len(values_b))),
        }

    return {
        "test_type": "insufficient_samples",
        "statistic": math.nan,
        "p_value": math.nan,
        "n_compared": int(min(len(values_a), len(values_b))),
    }
'''
    start = text.index("def _paired_or_independent_test(")
    end = text.index("\n\ndef _select_comparisons", start)
    script.write_text(text[:start] + replacement + text[end:], encoding="utf-8")
    print("Patched significance_tests.py duplicate-seed handling.")



# Cost guard: compute the exact training skip paths used by run_paper_optionA.py.
patch_significance_tests()
from scripts import run_paper_optionA as optiona

all_training_checkpoints = []
for config in (*optiona.ABLATION_CONFIGS, *optiona.MAIN_RESULTS_CONFIGS, *optiona.BASELINE_CONFIGS):
    experiment = optiona._load_experiment_name(Path(config))
    for seed in (3407, 3408):
        run_name = f"{experiment}_seed{seed}"
        all_training_checkpoints.append(Path("outputs") / experiment / run_name / "checkpoints" / "best.pt")

missing_training = [path for path in all_training_checkpoints if not path.exists()]
unexpected_missing = [path for path in missing_training if path not in ALLOWED_COLAB_TRAIN_CHECKPOINTS]
allowed_missing = [path for path in missing_training if path in ALLOWED_COLAB_TRAIN_CHECKPOINTS]

print("Option-A training checkpoint audit:")
print(f"  total possible training runs: {len(all_training_checkpoints)}")
print(f"  checkpoints already present: {len(all_training_checkpoints) - len(missing_training)}")
print(f"  allowed Colab training runs: {len(allowed_missing)}")
for path in allowed_missing:
    print(f"    ALLOW TRAIN: {path}")

if unexpected_missing:
    print("Unexpected missing checkpoints that would trigger extra A100 training:")
    for path in unexpected_missing:
        print(f"    BLOCK: {path}")
    if not ALLOW_OPTIONA_WITH_UNEXPECTED_TRAINING:
        raise RuntimeError(
            "Refusing to start A100 run because imports are incomplete. "
            "Re-run Section 6, verify Kaggle archives are on Drive, or set "
            "ALLOW_OPTIONA_WITH_UNEXPECTED_TRAINING=True knowingly."
        )

# Visibility guard: evaluations are inference, not training, but they still use GPU time.
missing_evaluations = []
for config in (*optiona.ABLATION_CONFIGS, *optiona.MAIN_RESULTS_CONFIGS, *optiona.BASELINE_CONFIGS):
    experiment = optiona._load_experiment_name(Path(config))
    for seed in (3407, 3408):
        run_name = f"{experiment}_seed{seed}"
        run_config = optiona._load_run_config(config, seed, tuple(COMMON_OVERRIDES), run_name)
        split = optiona._preferred_eval_split(run_config)
        dataset_name = run_config["dataset"]["name"]
        metrics_path = optiona._evaluation_json_path(experiment, run_name, dataset_name, split)
        if not metrics_path.exists():
            missing_evaluations.append(metrics_path)

print(f"  evaluation/inference JSONs missing: {len(missing_evaluations)}")
for path in missing_evaluations[:40]:
    print(f"    EVAL: {path}")
if len(missing_evaluations) > 40:
    print(f"    ... {len(missing_evaluations) - 40} more")

if AUDIT_ONLY:
    raise SystemExit("AUDIT_ONLY=True: stopping before sync/training.")

sync_cmd = (
    "while true; do "
    f"  rsync -a outputs/ {DRIVE_BACKUP}/ >> /content/rsync.log 2>&1; "
    "  sleep 600; "
    "done"
)
sync_proc = subprocess.Popen(["bash", "-c", sync_cmd], start_new_session=True)
print(f"Started periodic Drive sync (PID {sync_proc.pid}); logs at /content/rsync.log")

try:
    run_checked([
        sys.executable,
        "scripts/run_paper_optionA.py",
        "--seeds", "3407", "3408",
        "--fail-fast",
        "--no-auto-resume",
        "--override",
        *COMMON_OVERRIDES,
    ])
finally:
    try:
        os.killpg(os.getpgid(sync_proc.pid), signal.SIGTERM)
        print("Stopped periodic Drive sync.")
    except ProcessLookupError:
        pass
    print("Final rsync to Drive...")
    run_checked(["rsync", "-a", "--info=progress2", "outputs/", f"{DRIVE_BACKUP}/"])


## 10. Continue qualitative paper figure production

Run this after Section 9 has produced or restored checkpoints/evaluation outputs. It is resumable: existing paper panels are reused, missing panels are generated with inference only, and the resulting contact sheets are backed up to Drive.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path(globals().get("REPO_DIR", "/content/WBSNet-research-paper"))
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
print("CWD:", Path.cwd())

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
PAPER_FIGURE_DIR = Path("outputs/paper_figures/qualitative")
PAPER_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def run_checked(cmd):
    cmd = [str(x) for x in cmd]
    print("\n>>> " + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


# If this cell is being run after a reconnect in a fresh runtime, pull back the
# completed outputs first. This is safe to rerun and does not train anything.
if DRIVE_BACKUP.exists():
    Path("outputs").mkdir(exist_ok=True)
    if not any(Path("outputs").rglob("checkpoints/best.pt")):
        print(f"Restoring outputs from {DRIVE_BACKUP}")
        run_checked(["rsync", "-a", "--info=progress2", f"{DRIVE_BACKUP}/", "outputs/"])
else:
    print(f"Drive backup not found at {DRIVE_BACKUP}; using local outputs only.")

PREDICT_OVERRIDES = [
    "dataset.split_strategy=pre_split_dirs",
    "dataset.num_workers=2",
    "dataset.prefetch_factor=2",
    "runtime.device=cuda",
    "runtime.wandb.mode=offline",
    "runtime.wandb.enabled=false",
    "evaluation.max_visualizations=8",
]

TARGETS = [
    {
        "label": "kvasir_wbsnet_seed3408",
        "config": "configs/kvasir_wbsnet.yaml",
        "experiment": "kvasir_wbsnet",
        "run_name": "kvasir_wbsnet_seed3408",
        "dataset_name": "Kvasir-SEG",
        "split_candidates": ["val", "test"],
    },
    {
        "label": "clinicdb_wbsnet_seed3408",
        "config": "configs/clinicdb_wbsnet.yaml",
        "experiment": "clinicdb_wbsnet",
        "run_name": "clinicdb_wbsnet_seed3408",
        "dataset_name": "CVC-ClinicDB",
        "split_candidates": ["val", "test"],
    },
    {
        "label": "isic2018_wbsnet_seed3408",
        "config": "configs/isic2018_wbsnet.yaml",
        "experiment": "isic2018_wbsnet",
        "run_name": "isic2018_wbsnet_seed3408",
        "dataset_name": "ISIC2018",
        "split_candidates": ["test", "val"],
    },
]

# Generalization panels are created by evaluate.py under the Kvasir run folder.
EXISTING_PANEL_DIRS = [
    {
        "label": "kvasir_to_colondb_seed3408",
        "path": Path("outputs/kvasir_wbsnet/kvasir_wbsnet_seed3408/evaluation/CVC-ColonDB_all/predictions"),
    },
    {
        "label": "kvasir_to_colondb_seed3407",
        "path": Path("outputs/kvasir_wbsnet/kvasir_wbsnet_seed3407/evaluation/CVC-ColonDB_all/predictions"),
    },
]


def has_panels(path: Path) -> bool:
    return path.exists() and any(path.glob("*_paper_panel.png"))


def candidate_panel_dirs(run_dir: Path, dataset_name: str, split_candidates: list[str]) -> list[Path]:
    dirs = [run_dir / "predictions"]
    dirs.extend(run_dir / "evaluation" / f"{dataset_name}_{split}" / "predictions" for split in split_candidates)
    dirs.extend(sorted((run_dir / "evaluation").glob("*/predictions")) if (run_dir / "evaluation").exists() else [])
    seen = set()
    unique_dirs = []
    for path in dirs:
        resolved = str(path)
        if resolved not in seen:
            unique_dirs.append(path)
            seen.add(resolved)
    return unique_dirs


def choose_split(split_candidates: list[str], dataset_root: Path) -> str:
    for split in split_candidates:
        split_root = dataset_root / split
        if (split_root / "images").exists() and (split_root / "masks").exists():
            return split
    return split_candidates[0]


figure_manifest = []

for target in TARGETS:
    run_dir = Path("outputs") / target["experiment"] / target["run_name"]
    ckpt = run_dir / "checkpoints" / "best.pt"
    if not ckpt.exists():
        print(f"SKIP {target['label']}: missing checkpoint {ckpt}")
        continue

    panel_dir = None
    for candidate_dir in candidate_panel_dirs(run_dir, target["dataset_name"], target["split_candidates"]):
        if has_panels(candidate_dir):
            panel_dir = candidate_dir
            print(f"Using existing panels for {target['label']}: {panel_dir}")
            break

    if panel_dir is None:
        panel_dir = run_dir / "predictions"
        dataset_roots = {
            "Kvasir-SEG": Path("data/Kvasir-SEG"),
            "CVC-ClinicDB": Path("data/CVC-ClinicDB"),
            "ISIC2018": Path("data/ISIC2018"),
        }
        split = choose_split(target["split_candidates"], dataset_roots[target["dataset_name"]])
        print(f"Generating missing panels for {target['label']} on split '{split}'")
        run_checked([
            sys.executable,
            "predict.py",
            "--config", target["config"],
            "--checkpoint", ckpt,
            "--split", split,
            "--output-dir", panel_dir,
            "--override",
            *PREDICT_OVERRIDES,
            f"experiment.run_name={target['run_name']}",
        ])

    if not has_panels(panel_dir):
        raise RuntimeError(f"No paper panels were produced for {target['label']} in {panel_dir}")

    output_path = PAPER_FIGURE_DIR / f"{target['label']}_contact_sheet.png"
    run_checked([
        sys.executable,
        "scripts/make_paper_figures.py",
        "--input-dir", panel_dir,
        "--output", output_path,
        "--limit", "8",
        "--columns", "2",
    ])
    figure_manifest.append((target["label"], output_path, panel_dir))

for item in EXISTING_PANEL_DIRS:
    panel_dir = item["path"]
    if not has_panels(panel_dir):
        print(f"SKIP {item['label']}: no existing panels at {panel_dir}")
        continue
    output_path = PAPER_FIGURE_DIR / f"{item['label']}_contact_sheet.png"
    run_checked([
        sys.executable,
        "scripts/make_paper_figures.py",
        "--input-dir", panel_dir,
        "--output", output_path,
        "--limit", "8",
        "--columns", "2",
    ])
    figure_manifest.append((item["label"], output_path, panel_dir))

manifest_path = PAPER_FIGURE_DIR / "manifest.txt"
manifest_path.write_text(
    "\n".join(f"{label}\t{sheet}\tfrom={source}" for label, sheet, source in figure_manifest) + "\n",
    encoding="utf-8",
)

print("\nGenerated/verified paper figure sheets:")
for label, sheet, source in figure_manifest:
    print(f"  {label}: {sheet} (source panels: {source})")

if not figure_manifest:
    raise RuntimeError("No paper figure sheets were generated. Check checkpoints and prediction folders under outputs/.")

if DRIVE_BACKUP.exists():
    print("\nBacking up paper figures and outputs to Drive...")
    run_checked(["rsync", "-a", "--info=progress2", "outputs/", f"{DRIVE_BACKUP}/"])
else:
    print("\nDrive backup root does not exist; skipped rsync.")


## 11. Save outputs back to Drive

Colab disks reset on disconnect. Always rsync after any major run.

In [ ]:
from pathlib import Path
import subprocess

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", "outputs/", f"{DRIVE_BACKUP}/"], check=True)
print(f"Backed up outputs to {DRIVE_BACKUP}")


## 12. Build the paper PDF (optional, can also be done locally)

In [ ]:
# Uncomment to install LaTeX and build the PDF inside Colab.
# !apt-get install -y --quiet texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-publishers
# !cd paper && pdflatex paper && bibtex paper && pdflatex paper && pdflatex paper